# Mini Project - Company GHG Inventory Report Day 10
***Topic: *** Full Scope 1-2-3 GHG Inventory
# Stage 1 - Data Loading




In [12]:
# Emission factors - KgCO2e per unit of activity
factors = {
    # Scope 1 - Direct emissions
    "diesel": 2.68,
    "natural_gas": 2.02,
    "petrol": 2.31,
    "refrigrant": 1430,
    
# Scope 2 - Indirect emissions
    "electricity": 0.42,
    "steam": 0.19,

# Scope 3 - Other indirect emissions
    "flight_long": 0.19,
    "hotel_night": 12.5,
    "car_commute": 0.17,
    "waste_landfill": 467,
    "purchased_steel": 1800,
}

# Activity Record - One row per line item
records = [
    # Scope 1
    {"id":"A001","scope":"1","activity":"diesel","amount":15400,"site":"Karachi HQ"},
    {"id":"A002","scope":"1","activity":"natural_gas","amount":22000,"site":"Karachi HQ"},
    {"id":"A003","scope":"1","activity":"refrigrant","amount":2.5,"site":"Lahore Office"},

    # Scope 2
    {"id":"A004","scope":"2","activity":"electricity","amount":48000,"site":"Karachi HQ"},
    {"id":"A005","scope":"2","activity":"electricity","amount":21000,"site":"Lahore Office"},

    # Scope 3
    {"id":"A006","scope":"3","activity":"flight_long","amount":18500,"site":"Business Travel"},
    {"id":"A007","scope":"3","activity":"hotel_night","amount":42,"site":"Business Travel"},
    {"id":"A008","scope":"3","activity":"car_commute","amount":68000,"site":"Employee Commuting"},
    {"id":"A009","scope":"3","activity":"waste_landfill","amount":8.4,"site":"Karachi HQ"},
    {"id":"A010","scope":"3","activity":"purchased_steel","amount":12,"site":"Procurement"},
]

s1 = [r for r in records if r["scope"] == "1"]
s2 = [r for r in records if r["scope"] == "2"]
s3 = [r for r in records if r["scope"] == "3"]
print(f"Loaded {len(records)} records (S1: {len(s1)}, S2: {len(s2)}, S3: {len(s3)})")

    

Loaded 10 records (S1: 3, S2: 2, S3: 5)


# Stage 2 - Data Validation 

In [13]:
valid, issues = [], []

for r in records:
    activity = r.get("activity")
    amount = r.get("amount", 0)
    if not activity:
        issues.append(f"{r["id"]}: missing activities")
    elif activity not in factors:
        issues.append(f"{r["id"]}: no factor for '{activity}'")
    elif amount <= 0:
        issues.append(f"{r["id"]}: invalid amount ({amount})")
    else:
        valid.append(r)

print(f"ALL Valid : {len(valid)}")
print(f"Issues : {len(issues)}")
for i in issues:
    print(f" - {i}")



ALL Valid : 10
Issues : 0


# Stage 3 - Calculating tCO2e

In [28]:
for r in valid:
    factor = factors[r["activity"]]
    r["tonnes"] = round(r["amount"] * factor / 1000, 3)

for r in valid:
    print(f" S{r["scope"]} {r["id"]} {r["activity"]:<15} {r["tonnes"]:>5.2f} ({r["site"]})")

 S1 A001 diesel          41.27 (Karachi HQ)
 S1 A002 natural_gas     44.44 (Karachi HQ)
 S1 A003 refrigrant       3.58 (Lahore Office)
 S2 A004 electricity     20.16 (Karachi HQ)
 S2 A005 electricity      8.82 (Lahore Office)
 S3 A006 flight_long      3.52 (Business Travel)
 S3 A007 hotel_night      0.53 (Business Travel)
 S3 A008 car_commute     11.56 (Employee Commuting)
 S3 A009 waste_landfill   3.92 (Karachi HQ)
 S3 A010 purchased_steel 21.60 (Procurement)


# Stage 4- Total by Scope

In [37]:
by_scope = {}
for r in valid:
    s = r["scope"]
    by_scope[s] = by_scope.get(s, 0) + r["tonnes"]

by_scope = {k: round(v, 2) for k, v in by_scope.items()}
grand_total = round(sum(by_scope.values()), 2)

print(f"Scope 1: {by_scope["1"]} tCO2e")
print(f"Scope 2: {by_scope["2"]} tCO2e")
print(f"Scope 3: {by_scope["3"]} tCO2e")

print("----------------------------")
print(f"Total: {grand_total} tCO2e")

Scope 1: 89.29 tCO2e
Scope 2: 28.98 tCO2e
Scope 3: 41.12 tCO2e
----------------------------
Total: 159.39 tCO2e


# Stage 5: Formatted GHG inventory report

In [62]:
title = "Indus Steel GHG Inventory Report"
print(title)
print("=" * len(title))
print("Reporting Entity :   Indus Group")
print("Reporting Period :   Jan 2025 - Dec 2025")
print("Boundry          :   Karachi HQ and Lahor Office")
print("Standard         :   GHG Protocol")
print()

print(f"{"Scope":<10} {"tCO2e":>10} {"% total":>12}")
print("-" * 35)

for s in sorted(by_scope):
    val = by_scope[s]
    pct = round(val / grand_total * 100, 1)
    print(f"scope {s} {val:>12}{pct:>12}%")

print("-" * 35)
print(f"{"Total":<10}{grand_total:>10}{"100.0":>11}%")
print()

biggest = max(valid, key=lambda r: r["tonnes"])
print(f" Biggest single emisstion activity: {biggest["activity"]}"
      f"at {biggest["site"]} {biggest["tonnes"]}")

Indus Steel GHG Inventory Report
Reporting Entity :   Indus Group
Reporting Period :   Jan 2025 - Dec 2025
Boundry          :   Karachi HQ and Lahor Office
Standard         :   GHG Protocol

Scope           tCO2e      % total
-----------------------------------
scope 1        89.29        56.0%
scope 2        28.98        18.2%
scope 3        41.12        25.8%
-----------------------------------
Total         159.39      100.0%

 Biggest single emisstion activity: natural_gasat Karachi HQ 44.44
